# Gold -- KPIs de Negocio Logistico
**LogiLake | D'LOGIA** -- Capa Gold del Medallion Architecture

KPIs calculados en esta capa:
- **OTIF Rate** (On Time In Full) -- metrica principal de logistica
- **Delivery Performance** -- dias reales vs estimados, retraso promedio
- **Revenue Analytics** -- total, ticket promedio, por categoria y estado
- **Cancellation Rate** -- tasa de cancelacion mensual
- **NPS Proxy** -- distribucion de review scores

In [ ]:
# SparkSession con Delta Lake 3.1.0 (PySpark 3.5.0)
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("logilake_gold")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Delta Lake activo")

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BASE_PATH   = "./data"
SILVER_PATH = f"{BASE_PATH}/silver/olist_orders"
GOLD_PATH   = f"{BASE_PATH}/gold"

os.makedirs(GOLD_PATH, exist_ok=True)

silver = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver: {silver.count():,} registros")

In [ ]:
# KPI 1: Resumen Global
kpi_global = silver.agg(
    F.count("order_id").alias("total_orders"),
    F.sum(F.col("is_delivered").cast("int")).alias("total_delivered"),
    F.sum(F.col("is_canceled").cast("int")).alias("total_canceled"),
    F.round(
        F.sum(
            F.when(F.col("is_delivered") & ~F.col("is_late_delivery"), 1).otherwise(0)
        ).cast("double") /
        F.nullif(F.sum(F.col("is_delivered").cast("int")), 0) * 100,
        2
    ).alias("otif_rate_pct"),
    F.round(F.avg(F.when(F.col("is_delivered"), F.col("delivery_days_actual"))), 1)
        .alias("avg_delivery_days_actual"),
    F.round(F.avg(F.when(F.col("is_delivered"), F.col("delivery_days_estimated"))), 1)
        .alias("avg_delivery_days_estimated"),
    F.round(F.avg(F.when(F.col("is_delivered"), F.col("delivery_delay_days"))), 1)
        .alias("avg_delay_days"),
    F.round(F.sum("payment_value"), 2).alias("total_revenue_brl"),
    F.round(F.avg("payment_value"), 2).alias("avg_order_value_brl"),
    F.round(F.avg("review_score"), 2).alias("avg_review_score"),
    F.round(F.avg("freight_ratio") * 100, 1).alias("avg_freight_ratio_pct"),
    F.current_timestamp().alias("gold_computed_at"),
)

kpi_global.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/kpi_global")
print("kpi_global OK")
kpi_global.show(truncate=False)

In [ ]:
# KPI 2: OTIF y Delivery por Mes
kpi_monthly = (
    silver
    .withColumn("order_month", F.date_format("order_purchase_ts", "yyyy-MM"))
    .groupBy("order_month")
    .agg(
        F.count("order_id").alias("orders"),
        F.sum(F.col("is_delivered").cast("int")).alias("delivered"),
        F.sum(F.col("is_canceled").cast("int")).alias("canceled"),
        F.round(
            F.sum(
                F.when(F.col("is_delivered") & ~F.col("is_late_delivery"), 1).otherwise(0)
            ).cast("double") /
            F.nullif(F.sum(F.col("is_delivered").cast("int")), 0) * 100,
            2
        ).alias("otif_rate_pct"),
        F.round(F.avg(F.when(F.col("is_delivered"), F.col("delivery_days_actual"))), 1)
            .alias("avg_delivery_days"),
        F.round(F.avg(F.when(F.col("is_delivered"), F.col("delivery_delay_days"))), 1)
            .alias("avg_delay_days"),
        F.round(F.sum("payment_value"), 2).alias("revenue_brl"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
    )
    .withColumn("cancellation_rate_pct",
        F.round(F.col("canceled") / F.col("orders") * 100, 2))
    .orderBy("order_month")
)

kpi_monthly.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/kpi_monthly")
print("kpi_monthly OK")
kpi_monthly.show(5)

In [ ]:
# KPI 3: Revenue por Categoria
kpi_category = (
    silver
    .withColumn("category", F.explode_outer(F.col("categories")))
    .filter(F.col("category").isNotNull())
    .groupBy("category")
    .agg(
        F.count("order_id").alias("orders"),
        F.round(F.sum("payment_value"), 2).alias("revenue_brl"),
        F.round(F.avg("payment_value"), 2).alias("avg_order_value"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
        F.round(
            F.sum(F.when(F.col("is_delivered") & ~F.col("is_late_delivery"), 1).otherwise(0))
            .cast("double") /
            F.nullif(F.sum(F.col("is_delivered").cast("int")), 0) * 100,
            2
        ).alias("otif_rate_pct"),
    )
    .orderBy(F.desc("revenue_brl"))
)

kpi_category.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/kpi_category")
print("kpi_category OK")
kpi_category.show(10)

In [ ]:
# KPI 4: Distribucion de Review Scores (NPS Proxy)
reviews_total = silver.filter(F.col("review_score").isNotNull()).count()

kpi_nps = (
    silver
    .filter(F.col("review_score").isNotNull())
    .withColumn("review_score_int", F.col("review_score").cast("integer"))
    .groupBy("review_score_int")
    .agg(
        F.count("order_id").alias("orders"),
        F.round(F.avg("payment_value"), 2).alias("avg_payment_value"),
        F.round(F.avg("delivery_days_actual"), 1).alias("avg_delivery_days"),
    )
    .withColumn("pct_of_total",
        F.round(F.col("orders") / reviews_total * 100, 2))
    .orderBy("review_score_int")
)

kpi_nps.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/kpi_nps")
print("kpi_nps OK")
kpi_nps.show()

In [ ]:
# KPI 5: Delivery Performance por Estado del Vendedor
kpi_seller_state = (
    silver
    .filter(F.col("is_delivered"))
    .withColumn("seller_state", F.explode_outer(F.col("seller_states")))
    .filter(F.col("seller_state").isNotNull())
    .groupBy("seller_state")
    .agg(
        F.count("order_id").alias("orders"),
        F.round(F.avg("delivery_days_actual"), 1).alias("avg_delivery_days"),
        F.round(F.avg("delivery_delay_days"), 1).alias("avg_delay_days"),
        F.round(
            F.sum(
                F.when(~F.col("is_late_delivery"), 1).otherwise(0)
            ).cast("double") / F.count("order_id") * 100,
            2
        ).alias("otif_rate_pct"),
        F.round(F.sum("payment_value"), 2).alias("revenue_brl"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
    )
    .orderBy(F.desc("orders"))
)

kpi_seller_state.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/kpi_seller_state")
print("kpi_seller_state OK")
kpi_seller_state.show(10)

In [ ]:
# Resumen Gold
gold_tables = [
    ("kpi_global",       "Metricas globales de toda la operacion"),
    ("kpi_monthly",      "OTIF, revenue y entrega por mes"),
    ("kpi_category",     "Revenue y OTIF por categoria de producto"),
    ("kpi_nps",          "Distribucion de review scores"),
    ("kpi_seller_state", "Performance de entrega por estado del vendedor"),
]

print("=== GOLD LAYER -- Tablas creadas ===")
for name, desc in gold_tables:
    path = f"{GOLD_PATH}/{name}"
    try:
        count = spark.read.format("delta").load(path).count()
        print(f"  OK  {name:<20} {count:>6} filas  -- {desc}")
    except Exception as e:
        print(f"  ERR {name:<20} ERROR: {e}")